# LexiNet Evaluation

We first evaluate the intrinsic performance measure: perplexity, for our trained-and-saved forward n-gram models, followed by the extrinsic Hangman game simulation using our full current `GameSimulator` setup.

The perplexity evaluator uses the standard forward likelihood formula: lower perplexity means the model assigns higher average probability to the observed letters. Because `train.py` skips `</s>` as a prediction target, this notebook also skips `</s>` targets by default.

In [1]:
from pathlib import Path
import multiprocessing as mp
import os
import platform
import sys
from time import perf_counter

import pandas as pd
from IPython.display import HTML, display

sys.dont_write_bytecode = True

# In Colab, set this to Path("/content/lexinet") or your Drive path if auto-discovery cannot find the repo.
PROJECT_ROOT_OVERRIDE = None


def find_project_root(start=None):
    if PROJECT_ROOT_OVERRIDE is not None:
        return Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "evaluate.py").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find the LexiNet project root.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluate import DEFAULT_N_VALUES, evaluate_model_set

In [3]:
## Configuration

## Use `MODEL_FILE_TEMPLATE` to choose which trained model run to evaluate. The current setting evaluates the May 16 run. To evaluate the default Kneser-Ney files, change it to `"n_{n}_gram_model_kneser_ney.pkl"`.

MODELS_DIR = PROJECT_ROOT / "results" / "models"
TRAIN_DATA_PATH = PROJECT_ROOT / "data" / "train" / "words_train.txt"
TEST_DATA_PATH = PROJECT_ROOT / "data" / "test" / "words_test.txt"

N_VALUES = DEFAULT_N_VALUES
MODEL_FILE_TEMPLATE = "n_{n}_gram_model_may16.pkl"

# Without a floor, any unseen test n-gram has probability 0 and perplexity becomes infinity.
FLOOR_VALUE = 1e-12
INCLUDE_END_TOKENS = False

# Perplexity

For the summary, we have 8 values: one train and one test perplexity score for the n-grams models with n= 3, 4, 5, and 6.

In [29]:
def format_perplexity(value):
    if pd.isna(value):
        return ""
    if value >= 1_000_000:
        return f"{value:.4e}"
    return f"{value:,.4f}"


started_at = perf_counter()
rows = evaluate_model_set(
    models_dir=MODELS_DIR,
    train_data_path=TRAIN_DATA_PATH,
    test_data_path=TEST_DATA_PATH,
    n_values=N_VALUES,
    model_file_template=MODEL_FILE_TEMPLATE,
    floor_value=FLOOR_VALUE,
    include_end_tokens=INCLUDE_END_TOKENS,
)
elapsed_seconds = perf_counter() - started_at

results_df = pd.DataFrame(rows)
results_df["model_name"] = results_df["model_file"].map(lambda path: Path(path).name)

display(f"Evaluation complete. Calculated {len(results_df)} perplexity values in {elapsed_seconds:.2f} seconds.")

'Evaluation complete. Calculated 8 perplexity values in 23.86 seconds.'

In [30]:
perplexity_table = (
    results_df
    .pivot(index="n", columns="split", values="perplexity")
    .reset_index()
    .loc[:, ["n", "train", "test"]]
)

display(
    perplexity_table
    .style
    .format({"train": format_perplexity, "test": format_perplexity})
    .set_caption("Forward n-gram perplexity by split")
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "18px"), ("font-weight", "700"), ("padding", "8px 0")]},
        {"selector": "th", "props": [("background", "#eef4fb"), ("color", "#172033"), ("font-weight", "700")]},
        {"selector": "td", "props": [("padding", "8px 12px")]},
    ])
)

split,n,train,test
0,3,10.1328,9.6800
1,4,7.9106,8.1751
2,5,5.8423,10.3064
3,6,4.4727,23.4646


In [31]:
# `zero_probability_count` tells how many evaluated events were unseen by that model and therefore used `FLOOR_VALUE`. On the training split this should usually be zero; on the test split it gives useful context for large perplexity values.

diagnostics = results_df[["n", "split", "perplexity", "average_negative_log_likelihood", "token_count", "zero_probability_count", "floor_value", "model_name",]].sort_values(["n", "split"])
display(
    diagnostics
    .style
    .format({
        "perplexity": format_perplexity,
        "average_negative_log_likelihood": "{:.6f}",
        "floor_value": "{:.0e}",
    })
    .set_caption("Perplexity diagnostics")
)

,n,split,perplexity,average_negative_log_likelihood,token_count,zero_probability_count,floor_value,model_name
1,3,test,9.6800,2.270067,1636288,344,1e-12,n_3_gram_model_may16.pkl
0,3,train,10.1328,2.315779,2124746,0,1e-12,n_3_gram_model_may16.pkl
3,4,test,8.1751,2.101092,1636288,5856,1e-12,n_4_gram_model_may16.pkl
2,4,train,7.9106,2.068204,2124746,0,1e-12,n_4_gram_model_may16.pkl
5,5,test,10.3064,2.332769,1636288,38371,1e-12,n_5_gram_model_may16.pkl
4,5,train,5.8423,1.765119,2124746,0,1e-12,n_5_gram_model_may16.pkl
7,6,test,23.4646,3.155493,1636288,102951,1e-12,n_6_gram_model_may16.pkl
6,6,train,4.4727,1.497989,2124746,0,1e-12,n_6_gram_model_may16.pkl


# Hangman-Game Challenge Extrinsic Evaluation through simulation

In [4]:

# Now we evaluate the current full game-playing setup: the simulator loads the trained 3, 4, 5, and 6-gram dictionaries, uses the current bidirectional `GreedyPlayer` logic, and plays every word in the configured splits.

# This section uses the multiprocessing support in `GameSimulator.simulate_games`. The notebook detects Colab vs macOS and picks a sensible default, but you can override `SIMULATION_WORKERS` and `SIMULATION_START_METHOD` manually.

from src.game_simulator import GameSimulator


SIMULATOR_RUN_NAME = "may16"
SIMULATOR_METHOD_NAME = "best"
MAX_LIVES = 6

IS_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
IS_MAC = platform.system() == "Darwin"
RUNTIME_ENVIRONMENT = "Colab" if IS_COLAB else "macOS" if IS_MAC else platform.system()
AVAILABLE_WORKERS = os.cpu_count() or 1
AVAILABLE_START_METHODS = mp.get_all_start_methods()

DEFAULT_SIMULATION_WORKERS = AVAILABLE_WORKERS if IS_COLAB else min(1, AVAILABLE_WORKERS) if IS_MAC else AVAILABLE_WORKERS
DEFAULT_START_METHOD = "fork" if IS_COLAB and "fork" in AVAILABLE_START_METHODS else None

SIMULATION_WORKERS = DEFAULT_SIMULATION_WORKERS
SIMULATION_START_METHOD = DEFAULT_START_METHOD
SIMULATION_CHUNKSIZE = None

# Keep this as None for the full train/test evaluation. Set an integer for quick smoke runs.
SIMULATION_SAMPLE_SIZE = None

SIMULATION_SPLITS = {
    "test": TEST_DATA_PATH,
    "train": TRAIN_DATA_PATH,
}

SAVE_SIMULATION_CSVS = False
SIMULATION_OUTPUT_DIR = PROJECT_ROOT / "results"

EFFECTIVE_WORKERS = AVAILABLE_WORKERS if SIMULATION_WORKERS is None else SIMULATION_WORKERS
START_METHOD_LABEL = SIMULATION_START_METHOD or "python default"

print('Runtime environment:', RUNTIME_ENVIRONMENT)
print('Run name for the models to be used:', SIMULATOR_RUN_NAME)
print('The methodology we are using is the:', SIMULATOR_METHOD_NAME)
print('Max lives for the hangman game challenge:', MAX_LIVES)
print('Workers used for simulations are:', EFFECTIVE_WORKERS, 'out of available count of', AVAILABLE_WORKERS) # mac is powerful no doubt

Runtime environment: macOS
Run name for the models to be used: may16
The methodology we are using is the: best
Max lives for the hangman game challenge: 6
Workers used for simulations are: 1 out of available count of 8


In [5]:
# running game simulations now!!!
# The summary table reports the headline extrinsic metric: win rate over each split. The detailed table below it keeps the per-word-length breakdown returned by the simulator.

def run_game_simulation(split_name, word_list_path):
    started_at = perf_counter()
    simulator = GameSimulator(
        str(word_list_path),
        str(MODELS_DIR),
        SIMULATOR_RUN_NAME,
        SIMULATOR_METHOD_NAME,
        max_lives=MAX_LIVES,
    )

    if SIMULATION_SAMPLE_SIZE is not None:
        simulator.word_list = simulator.word_list[:SIMULATION_SAMPLE_SIZE]

    output_csv_path = None
    if SAVE_SIMULATION_CSVS:
        output_csv_path = SIMULATION_OUTPUT_DIR / f"game_results_{split_name}_{SIMULATOR_RUN_NAME}.csv"

    num_wins, total_games, results_by_length, results_df = simulator.simulate_games(
        output_csv_path=str(output_csv_path) if output_csv_path else None,
        n_workers=SIMULATION_WORKERS,
        chunksize=SIMULATION_CHUNKSIZE,
        multiprocessing_start_method=SIMULATION_START_METHOD,
    )
    elapsed_seconds = perf_counter() - started_at

    split_results_df = results_df.copy()
    split_results_df.insert(0, "split", split_name)

    summary = {
        "split": split_name,
        "total_games": total_games,
        "wins": num_wins,
        "losses": total_games - num_wins,
        "win_rate": round((num_wins / total_games) * 100, 2) if total_games else 0,
        "runtime": RUNTIME_ENVIRONMENT,
        "workers": EFFECTIVE_WORKERS,
        "start_method": START_METHOD_LABEL,
        "sample_size": SIMULATION_SAMPLE_SIZE if SIMULATION_SAMPLE_SIZE is not None else "full",
        "seconds": elapsed_seconds,
    }
    return summary, split_results_df


simulation_summaries = []
simulation_detail_tables = []

for split_name, word_list_path in SIMULATION_SPLITS.items():
    summary, split_results_df = run_game_simulation(split_name, word_list_path)
    simulation_summaries.append(summary)
    simulation_detail_tables.append(split_results_df)

extrinsic_summary_df = pd.DataFrame(simulation_summaries)
extrinsic_by_length_df = pd.concat(simulation_detail_tables, ignore_index=True)

display(
    extrinsic_summary_df
    .style
    .format({"win_rate": "{:.2f}%", "seconds": "{:.2f}"})
    .set_caption("Extrinsic game-simulation summary")
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "18px"), ("font-weight", "700"), ("padding", "8px 0")]},
        {"selector": "th", "props": [("background", "#eef4fb"), ("color", "#172033"), ("font-weight", "700")]},
        {"selector": "td", "props": [("padding", "8px 12px")]},
    ])
)

{1: 3, 2: 3, 3: 4, 4: 6, 5: 6, 6: 6, 7: 6, 8: 6, 9: 6, 10: 6, 11: 6, 12: 6, 13: 6, 14: 6, 15: 6, 16: 6, 17: 6, 18: 6, 19: 6, 20: 6, 21: 6, 22: 6, 23: 6, 24: 6, 25: 6, 26: 6, 27: 6, 28: 6, 29: 6, 30: 6, 31: 6, 32: 6, 33: 6, 34: 6, 35: 6, 36: 6, 37: 6, 38: 6, 39: 6, 40: 6, 41: 6, 42: 6, 43: 6, 44: 6, 45: 6, 46: 6, 47: 6, 48: 6, 49: 6}
num workers 1 code is running successfully right...


170671it [09:57, 285.83it/s]


{1: 3, 2: 3, 3: 4, 4: 6, 5: 6, 6: 6, 7: 6, 8: 6, 9: 6, 10: 6, 11: 6, 12: 6, 13: 6, 14: 6, 15: 6, 16: 6, 17: 6, 18: 6, 19: 6, 20: 6, 21: 6, 22: 6, 23: 6, 24: 6, 25: 6, 26: 6, 27: 6, 28: 6, 29: 6, 30: 6, 31: 6, 32: 6, 33: 6, 34: 6, 35: 6, 36: 6, 37: 6, 38: 6, 39: 6, 40: 6, 41: 6, 42: 6, 43: 6, 44: 6, 45: 6, 46: 6, 47: 6, 48: 6, 49: 6}
num workers 1 code is running successfully right...


227300it [30:42, 123.39it/s]


,split,total_games,wins,losses,win_rate,runtime,workers,start_method,sample_size,seconds
0,test,170671,109686,60985,64.27%,macOS,1,python default,full,603.11
1,train,227300,149879,77421,65.94%,macOS,1,python default,full,799.24


In [ ]:

# Extrinsic game-simulation summary
#  	split	total_games	wins	losses	win_rate	runtime	workers	start_method	sample_size	seconds
# 0	test	170671	109693	60978	64.27%	macOS	1	python default	full	606.36
# 1	train	227300	149879	77421	65.94%	macOS	1	python default	full	790.08

In [6]:
## Game Results By Word Length

## This keeps the simulator's existing by-length view, split by train/test. The total rows are shown in the summary above, so this table focuses on word lengths.

by_length_display = extrinsic_by_length_df[extrinsic_by_length_df["length"] != "Total"].copy()
by_length_display["length"] = by_length_display["length"].astype(int)
by_length_display = by_length_display.sort_values(["split", "length"])

display(
    by_length_display
    .style
    .format({"win_rate": "{:.2f}%"})
    .set_caption("Extrinsic game-simulation results by word length")
)

,split,length,total,wins,win_rate
22,test,1,2,1,50.00%
25,test,2,41,10,24.39%
28,test,3,531,60,11.30%
27,test,4,2580,378,14.65%
26,test,5,6456,1268,19.64%
24,test,6,13112,3757,28.65%
23,test,7,19156,7507,39.19%
21,test,8,24119,12554,52.05%
20,test,9,24495,15700,64.09%
19,test,10,21657,15882,73.33%
